# Week 06 - Image Preprocessing II: Filtering, Restoration and Morphology

**MCTE 4323 / MCTA 4364 Machine Vision**

### Learning objectives
By the end of this lab you will be able to:
- Explain **convolution** and apply linear filters with `cv2.filter2D`.
- Compare **linear** (mean, Gaussian) and **non-linear** (median, bilateral) filters.
- Restore images corrupted by Gaussian and salt-and-pepper noise.
- Detect edges with **Sobel** and **Laplacian** and sharpen with unsharp masking.
- Apply **mathematical morphology**: erosion, dilation, opening, closing, gradient, top-hat.

### Convolution core idea
A filter (kernel) slides over the image and computes a weighted sum:

$$g(x,y) = \sum_{i,j} k(i,j)\, f(x-i, y-j)$$

The kernel design determines the effect: averaging removes noise, derivatives find edges, and subtracting a smoothed image sharpens.

## 1. Setup

In [ ]:
import os

# Works in Colab (clones the repo) and locally or in CI (runs inside the repo)
if not os.path.exists("resources/scripts/cvhelpers.py"):
    if not os.path.isdir("MCTA-4364-Machine-Vision"):
        !git clone https://github.com/hasanzaki/MCTA-4364-Machine-Vision.git
    %cd MCTA-4364-Machine-Vision
!pip -q install opencv-python numpy matplotlib ipywidgets


In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

def show(*images, titles=None, cmap=None):
    titles = titles or [""] * len(images)
    plt.figure(figsize=(5 * len(images), 5))
    for i, img in enumerate(images):
        plt.subplot(1, len(images), i + 1)
        plt.imshow(img, cmap=cmap or (None if img.ndim == 3 else "gray"))
        plt.title(titles[i]); plt.axis("off")
    plt.tight_layout(); plt.show()

img = cv2.imread("resources/images/hasan.jpg")
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
print("Shape:", img.shape)

## 2. Guided example - convolution by hand vs `filter2D`
We implement convolution naively for a small region to see the arithmetic, then verify against OpenCV's optimised `filter2D`.

In [ ]:
# 3x3 box (mean) kernel. Note: cv2.filter2D performs correlation.
kernel = np.ones((3, 3), np.float32) / 9.0

# Manual correlation on a small patch
patch = gray[100:103, 100:103].astype(np.float32)
manual = np.sum(patch * kernel)

filtered = cv2.filter2D(gray, -1, kernel)
print("Manual value at (101,101):", round(float(manual), 2))
print("filter2D  value at (101,101):", int(filtered[101, 101]))

## 3. Guided example - linear smoothing filters
Larger kernels mean more blur. Gaussian weighting preserves structure better than a flat box filter.

In [ ]:
mean5 = cv2.blur(gray, (5, 5))
gauss5 = cv2.GaussianBlur(gray, (5, 5), sigmaX=1.5)
gauss11 = cv2.GaussianBlur(gray, (11, 11), sigmaX=3.0)
show(gray, mean5, gauss5, gauss11, titles=["Original", "Mean 5x5", "Gaussian 5x5", "Gaussian 11x11"])

## 4. Guided example - non-linear filters
- **Median** removes salt-and-pepper noise while preserving edges.
- **Bilateral** smooths flat areas but keeps edges sharp (edge-preserving).

In [ ]:
median = cv2.medianBlur(gray, 5)
bilateral = cv2.bilateralFilter(gray, d=9, sigmaColor=75, sigmaSpace=75)
show(gray, median, bilateral, titles=["Original", "Median 5", "Bilateral"])

## 5. Guided example - noise and restoration
We corrupt the image with two common noise models and compare which filter restores it best.

In [ ]:
# Gaussian noise
gaussian_noise = np.random.normal(0, 25, gray.shape).astype(np.float32)
noisy_gauss = np.clip(gray.astype(np.float32) + gaussian_noise, 0, 255).astype(np.uint8)

# Salt-and-pepper noise
noisy_sp = gray.copy()
prob = 0.05
rnd = np.random.rand(*gray.shape)
noisy_sp[rnd < prob / 2] = 0
noisy_sp[rnd > 1 - prob / 2] = 255

show(noisy_gauss, cv2.GaussianBlur(noisy_gauss, (5, 5), 1.5),
     noisy_sp, cv2.medianBlur(noisy_sp, 5),
     titles=["Gaussian noise", "Gaussian blur", "Salt & pepper", "Median filter"])

> **Key result:** median filtering is far better for salt-and-pepper noise; Gaussian blur is better for Gaussian noise but softens edges.

## 6. Guided example - derivative filters and sharpening
Sobel approximates the first derivative (edges). The gradient magnitude combines both directions. Subtracting a blurred copy from the original is **unsharp masking**.

In [ ]:
sobel_x = cv2.Sobel(gray, cv2.CV_32F, 1, 0, ksize=3)
sobel_y = cv2.Sobel(gray, cv2.CV_32F, 0, 1, ksize=3)
mag = cv2.magnitude(sobel_x, sobel_y)
mag = cv2.normalize(mag, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)

laplacian = cv2.Laplacian(gray, cv2.CV_32F)
lap_vis = cv2.normalize(np.abs(laplacian), None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)

# Unsharp mask
blurred = cv2.GaussianBlur(gray, (0, 0), 3)
sharp = cv2.addWeighted(gray, 1.5, blurred, -0.5, 0)

show(mag, lap_vis, sharp, titles=["Sobel gradient magnitude", "Laplacian", "Unsharp sharpened"])

## 7. Guided example - mathematical morphology
Morphology works on shapes using a **structuring element**.
- **Erosion**: shrinks bright regions (removes small noise)
- **Dilation**: grows bright regions (fills gaps)
- **Opening** = erosion then dilation (removes small blobs)
- **Closing** = dilation then erosion (fills small holes)
- **Gradient** = dilation - erosion (outline)

In [ ]:
_, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
se = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))

eroded = cv2.erode(binary, se)
dilated = cv2.dilate(binary, se)
opened = cv2.morphologyEx(binary, cv2.MORPH_OPEN, se)
closed = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, se)
gradient = cv2.morphologyEx(binary, cv2.MORPH_GRADIENT, se)
show(binary, eroded, dilated, opened, closed, gradient,
     titles=["Binary", "Erode", "Dilate", "Open", "Close", "Gradient"])

## 8. Exercise (complete the code)

Build a **cleaning pipeline** for a noisy binary mask:
1. Start from `binary` above.
2. Add salt-and-pepper noise to it.
3. Remove the noise with `cv2.medianBlur` (kernel 5).
4. Clean remaining specks with `cv2.morphologyEx(..., cv2.MORPH_OPEN, se)`.
5. Count the connected components before and after using `cv2.connectedComponents`.

In [ ]:
# TODO: implement the cleaning pipeline


## 9. Challenge (independent)

Load `resources/images/money_counter.png`. Build a pipeline that:
1. Converts to grayscale and blurs to reduce noise.
2. Thresholds (Otsu).
3. Applies **morphological opening** to remove noise and **closing** to fill holes.
4. Counts the resulting bright objects with `cv2.connectedComponentsWithStats` (ignore tiny regions).

Report the number of objects detected.

In [ ]:
# Your code here


## 10. Reflection
1. Why does the median filter preserve edges better than a mean filter?
2. What is the difference between opening and closing in terms of noise and holes?
3. A production line uses a camera with salt-and-pepper noise. Which filter would you deploy and why?

## 11. Visual summary

In [ ]:
import sys
sys.path.append("resources/scripts")
from cvhelpers import concept_map

concept_map([
    "Noisy / raw image",
    "Linear filters: mean, Gaussian (smooth, blur edges)",
    "Non-linear filters: median (salt/pepper), bilateral (edge-preserving)",
    "Derivative filters: Sobel, Laplacian (edges)",
    "Sharpening: unsharp masking",
    "Morphology: erode/dilate -> open/close -> clean shapes"
], title="Filtering and morphology pipeline")

## 12. Interactive exploration - kernel size and filter type

Compare how the mean and median filters behave as the kernel grows. Which one keeps the edges crisp?

In [ ]:
import ipywidgets as widgets
from ipywidgets import interact

def filter_demo(kernel=5):
    k = kernel if kernel % 2 == 1 else kernel + 1
    mean = cv2.blur(gray, (k, k))
    median = cv2.medianBlur(gray, k)
    show(mean, median, titles=[f"mean {k}x{k}", f"median {k}"])

interact(filter_demo, kernel=widgets.IntSlider(min=3, max=15, step=2, value=5))

## 13. Check your understanding (Q&A)

<details><summary><b>Q1. Why does the median filter remove salt-and-pepper noise so well?</b></summary>

Isolated extreme pixels are never the median of their neighbourhood, so the median simply ignores them while keeping edges.

</details>

<details><summary><b>Q2. What is the difference between opening and closing?</b></summary>

Opening (erode then dilate) removes small bright blobs and breaks thin connections. Closing (dilate then erode) fills small holes and connects nearby regions.

</details>

<details><summary><b>Q3. Why does a bilateral filter preserve edges while a Gaussian does not?</b></summary>

The bilateral filter adds a range term: pixels with very different intensity contribute little, so smoothing stops at edges.

</details>

## 14. Further reading & self-exploration
- OpenCV image smoothing: https://docs.opencv.org/4.x/d4/d13/tutorial_py_filtering.html
- OpenCV morphology: https://docs.opencv.org/4.x/d9/d61/tutorial_py_morphological_ops.html
- OpenCV images restoration: https://docs.opencv.org/4.x/d5/d69/tutorial_py_non_local_means.html
- scikit-image filters: https://scikit-image.org/docs/stable/api/skimage.filters.html
- Wikipedia - Median filter: https://en.wikipedia.org/wiki/Median_filter

**Try next:** build an adaptive denoising pipeline that picks the filter automatically from the estimated noise.

## 15. Key takeaways
- Convolution is the core linear-filtering operation.
- Mean/Gaussian smooth; median removes impulse noise; bilateral preserves edges.
- Sobel/Laplacian detect edges; unsharp masking sharpens.
- Morphology cleans masks: open removes specks, close fills holes.
